In [ ]:
import time
import json

# ! will NEED PyTorch version 2.5+ --> for scaled_dot_product_attention(enable_gqa=True)
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple
from einops import rearrange
from model_config import ModelConfig, AttentionConfig, FFNConfig
import training_engine as training_engine 
from tensor_tools import init_custom_weight
from inference_engine import advanced_inference, load_model_weights
from kv_cache import KV_cache


device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device:         {device}")
print(f"PyTorch:        {torch.__version__}")

if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"CUDA version:   {torch.version.cuda}")

    print(f"device capability: {torch.cuda.get_device_capability()}")
    print(f"flash_attention_available: {torch.backends.cuda.is_flash_attention_available()}")




Device:         cuda
PyTorch:        2.8.0+cu129
GPU:            NVIDIA GeForce GTX 1660 Ti
CUDA version:   12.9
device capability: (7, 5)
flash_attention_available: False


In [ ]:

configs = ModelConfig(

    model_name="level3_FFN_models",
    tied_embeddings=False,  
    
    embed_dim=2**5,   # 32
    vocab_size=2**11,  # 2048
    num_layers=5,
    max_training_batch_size=2**4,   # 16
    max_context_window=2**6,   # 64    
    learning_rate=1e-3,

    attention=AttentionConfig(
        num_q_heads =2**0,  # 1 | 1   x   x    x
        num_kv_heads=2**0,  # 1 | 1   1   2+   x
        rope_theta  =10000.0, 
    ),
    
    ffn=FFNConfig(
        mlp_to_embed_expand_factor= 2**2,
        use_gate=True,
        
        Moe_num_experts= 2**0, # dense if 1, MoE if more
        MoE_num_experts_per_tkn=2**0,
        MoE_capacity_factor=1.25,
        MoE_router_aux_loss_coef=0.01,
        MoE_router_z_loss_coeff=0.0
    ),

    save_path       = "../models/level3_FFN_models.pt",
    val_loss_history= "../models/level3_FFN_models_val_loss_history.txt",

    checkpoint_dir= "../checkpoints/",
    checkpoint_every_steps= 250,
    keep_last_n_checkpoints= 3,

)

print(configs)
print(configs.head_dim)
print(configs.ffn_hidden_dim)

ModelConfig(model_name='level3_FFN_models', tied_embeddings=False, max_training_batch_size=16, embed_dim=32, vocab_size=2048, max_context_window=64, num_layers=5, learning_rate=0.001, attention=AttentionConfig(num_q_heads=1, num_kv_heads=1, rope_theta=10000.0), ffn=FFNConfig(use_gate=True, mlp_to_embed_expand_factor=4, num_experts=1, num_experts_per_tkn=1), save_path='../models/level3_FFN_models.pt', val_loss_history='../models/level3_FFN_models_val_loss_history.txt', checkpoint_dir='../checkpoints/', checkpoint_every_steps=250, keep_last_n_checkpoints=3)
32
128


In [ ]:
# RoPE

class RotaryEmbedding(nn.Module):

    def __init__(self, cfg:ModelConfig):
        super().__init__()

        base     = cfg.attention.rope_theta
        head_dim = cfg.embed_dim // cfg.attention.num_q_heads


        # shape: [head_dim // 2]
        inv_freq = 1.0 / (
            base ** (torch.arange(0, head_dim, 2).float() / head_dim)
        ).to(dtype=torch.float32)
        
        self.register_buffer("inv_freq", inv_freq, persistent=False)


    def forward(
            self,
            position_ids: torch.Tensor,   # [B, seq] or [seq]
            dtype= torch.float32,
    ) -> tuple[torch.Tensor, torch.Tensor]:
            
            """position_ids: [B, T] (or [T]) -> cos, sin of shape [B, 1, T, head_dim]"""

            if position_ids.dim() == 1:
                position_ids = position_ids.unsqueeze(0)          # [1, seq]


            # position_ids: [B, seq]  (integer)
            # inv_freq:     [head_dim//2]
            # freqs:        [B, seq, head_dim//2]
            freqs = torch.einsum(
                "bi,j->bij",
                position_ids.to(dtype=self.inv_freq.dtype),
                self.inv_freq,
            )

            emb = torch.cat((freqs, freqs), dim=-1) # [B, seq, head_dim]

            # broadcast over the head dimension → [B, 1, seq, head_dim]
            return emb.cos().to(dtype).unsqueeze(1), emb.sin().to(dtype).unsqueeze(1)


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(
    q:   torch.Tensor,
    k:   torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
    
    cos = cos.to(device=q.device, dtype=q.dtype)
    sin = sin.to(device=q.device, dtype=q.dtype)

    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k

In [ ]:
# Attention

class Attention_SubBlock(nn.Module):

    def __init__(self, cfg:ModelConfig):
        super().__init__()

        self.embed_dim    = cfg.embed_dim
        self.num_q_heads  = cfg.attention.num_q_heads
        self.num_kv_heads = cfg.attention.num_kv_heads
        self.num_layers   = cfg.num_layers
        
        if self.embed_dim % self.num_q_heads != 0:
            raise ValueError("embed_dim must be divisible by num_q_heads")
        if self.num_q_heads % self.num_kv_heads != 0:
            raise ValueError("num_q_heads must be divisible by num_kv_heads")

        self.head_dim = self.embed_dim // self.num_q_heads
        self.q_dim    = self.num_q_heads * self.head_dim
        self.kv_dim   = self.num_kv_heads * self.head_dim

        # ONE matrix for Q, K and V: columns are [ q | k | v ]
        self.W_qkv = nn.Parameter(torch.empty(self.embed_dim, self.q_dim + 2 * self.kv_dim))
        self.W_o   = nn.Parameter(torch.empty(self.q_dim, self.embed_dim))

        self.init_params()



    def init_params(self):

        q_end = self.q_dim
        k_end = q_end + self.kv_dim

        W_q = self.W_qkv[:, :q_end]
        W_k = self.W_qkv[:, q_end:k_end]
        W_v = self.W_qkv[:, k_end:]

        init_custom_weight(W_q, is_residual_output=False)
        init_custom_weight(W_k, is_residual_output=False)
        init_custom_weight(W_v, is_residual_output=False)

        # Residual output projection (scaled by depth)
        init_custom_weight(self.W_o, 
                           is_residual_output=True, 
                           num_layers=self.num_layers)


    def forward(
        self,
        x:   torch.Tensor,
        cos: torch.Tensor,          # [B, 1, seq, head_dim] – already computed
        sin: torch.Tensor,
        cache: Optional[KV_cache] = None,
        layer_idx: int = 0,
        
        lane_ids:  Optional[torch.Tensor] = None,
        attn_mask: Optional[torch.Tensor] = None,   # [B, 1, T, kv_len] bool, from cache.prepare()
   
    ) -> torch.Tensor:

        batch_size, seq_len, embed_dim = x.shape

        qkv = x @ self.W_qkv  # [B, T, q_dim + 2*kv_dim]
        q, k, v = qkv.split([self.q_dim, self.kv_dim, self.kv_dim], dim=-1)


        q = rearrange(q, 
                      "batch_size seq_len (num_q_heads head_dim)" 
                      " -> " 
                      "batch_size num_q_heads seq_len head_dim", 
                      num_q_heads=self.num_q_heads)
        
        k = rearrange(k, 
                      "batch_size seq_len (num_kv_heads head_dim)" 
                      " -> " 
                      "batch_size num_kv_heads seq_len head_dim",
                      num_kv_heads=self.num_kv_heads)
        
        v = rearrange(v, 
                      "batch_size seq_len (num_kv_heads head_dim)" 
                      " -> " 
                      "batch_size num_kv_heads seq_len head_dim",
                      num_kv_heads=self.num_kv_heads)


        q, k = apply_rotary_pos_emb(q, k, cos, sin)


        if cache is not None:
            k, v = cache.update(layer_idx, lane_ids, k, v)



        # The mask rule is that a query at absolute position p may attend to a key at 
        # absolute position a iff p - W < a <= p. The <= p part is causality, 
        # and the > p - W part is the sliding window. 
        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            is_causal=(attn_mask is None), # fast path only when no mask is given
            enable_gqa=(self.num_q_heads != self.num_kv_heads),
        )


        # Merge attention heads
        out = rearrange(
            out,
            "batch_size num_q_heads seq_len head_dim"
            " -> " 
            "batch_size seq_len (num_q_heads head_dim)",
        )


        return out @ self.W_o


* every FFN sub-block maps [B, T, D] -> ([B, T, D], aux_loss)

* aux_loss is a float32 scalar tensor. It is 0 for the dense blocks\
     and the (already coefficient-scaled) router loss for the MoE

In [ ]:
# FFN helper functions 


def _zero_aux(x: torch.Tensor) -> torch.Tensor:
    return x.new_zeros((), dtype=torch.float32)


def _ffn_activation(h: torch.Tensor, gated: bool) -> torch.Tensor:
    """h: [..., 2*H] ([gate | up]) if gated else [..., H]  ->  [..., H]"""
    if gated:
        gate, up = h.chunk(2, dim=-1)
        return F.silu(gate) * up
    return F.silu(h)


In [ ]:
class MLP_SubBlock(nn.Module):
    """Position-wise FFN: D -> H -> D with SiLU. Each token is processed independently."""

    def __init__(self, cfg:ModelConfig):
        super().__init__()

        self.num_layers = cfg.num_layers

        self.W_up   = nn.Parameter(torch.empty(
            cfg.embed_dim, cfg.ffn_hidden_dim
            )
        )
        
        self.W_down = nn.Parameter(torch.empty(
            cfg.ffn_hidden_dim, cfg.embed_dim
            )
        )

        self._init_weights()


    def _init_weights(self):
        init_custom_weight(self.W_up,   is_residual_output=False)
        
        init_custom_weight(self.W_down, is_residual_output=True , 
                                        num_layers=self.num_layers)

  
    def forward(self, x:torch.Tensor)-> Tuple[torch.Tensor, torch.Tensor]:
        out = _ffn_activation(x @ self.W_up, gated=False) @ self.W_down
        return out, _zero_aux(x)
        


In [ ]:
class GLU_SubBlock(nn.Module):
    """Gated FFN (SwiGLU). Params: W_gate_up [D, 2H] (= [gate | up]) and W_down [H, D]."""

    def __init__(self, cfg:ModelConfig):
        super().__init__()

        self.num_layers = cfg.num_layers
        self.hidden_dim = cfg.ffn_hidden_dim

        self.W_gate_up  = nn.Parameter(torch.empty(
            cfg.embed_dim, 2 * self.hidden_dim
            )
        )

        
        self.W_down = nn.Parameter(torch.empty(
            self.hidden_dim, cfg.embed_dim
            )
        )


        self._init_weights()


    def _init_weights(self):
        H = self.hidden_dim

        # initialise the two halves separately so each sees a [D, H] matrix
        init_custom_weight(self.W_gate_up[:, :H],   is_residual_output=False) # gate
        init_custom_weight(self.W_gate_up[:, H:],   is_residual_output=False) # up

        init_custom_weight(self.W_down, is_residual_output=True, 
                                        num_layers=self.num_layers)


    def forward(self, x):
        out = _ffn_activation(x @ self.W_gate_up, gated=True) @ self.W_down
        return out, _zero_aux(x)

In [ ]:
class Mixture_of_Experts_SubBlock(nn.Module):

    """
    Sparse Mixture-of-Experts FFN: every token is processed by only `top_k` of the
    `num_experts` expert FFNs, chosen by a learned router. Parameters grow with
    num_experts, compute per token only with top_k.
 
    Implementation = "capacity-based dense dispatch" (GShard / Switch Transformer style):
 
      1. ROUTE      router logits -> softmax -> top-k experts + gate weights per token.
      2. ASSIGN     give each (token, expert) pair a slot  0..C-1  inside that expert's buffer
                    (C = expert capacity). Pairs that don't fit are dropped.
      3. DISPATCH   scatter tokens into ONE dense buffer  [num_experts, C, embed_dim]   (index_add, no Python loop)
      4. COMPUTE    all experts at once with two batched matmuls (torch.bmm) over stacked
                    3-D weights  [num_experts, embed_dim, H] / [num_experts, H, embed_dim]   (no per-expert Python loop)
      5. COMBINE    gather every pair's output back, weight by its gate, sum over the k slots.
 
    Why this design
      * No data-dependent shapes and no `.item()` / `.tolist()` host syncs: shapes depend
        only on (B*T, num_experts, k, capacity_factor) -> torch.compile / CUDA-graph friendly.
      * Two big bmm calls instead of num_experts small matmuls -> far fewer kernel launches.
      * FLOPs ~ capacity_factor * (dense FFN FLOPs) * top_k, independent of num_experts.
 
    Capacity / token dropping
      C = ceil(capacity_factor * N * top_k / num_experts)   with N = B*T tokens in this call.
      * TRAINING with capacity_factor set: overflowing pairs are dropped (their expert
        contributes 0; the residual connection still carries the token) and priority goes
        to 1st-choice assignments over 2nd-choice, then to earlier tokens.
      * EVAL, or capacity_factor=None: C = N, which can never overflow (one token picks each
        expert at most once) -> exact, drop-free output. It wastes padding FLOPs, but at
        inference N is small (a few lanes x chunk), so that is the right trade-off, and it
        keeps results independent of what else is in the batch.
 
    Load balancing (Switch Transformer): aux = num_experts * sum_e f_e * P_e
        f_e = fraction of (token, slot) assignments routed to expert e   (not differentiable)
        P_e = mean router probability of expert e                        (differentiable)
      Minimised (=1.0) when both are uniform. Optional router z-loss (ST-MoE) penalises large
      router logits for numerical stability.
 
    Optional FFNConfig fields (read with getattr, so your current config keeps working):
        capacity_factor        (default 1.25, None -> never drop)
        router_aux_loss_coef   (default 0.01)
        router_z_loss_coef     (default 0.0)
    """


    def __init__(self, cfg:ModelConfig):
        super().__init__()

        self.embed_dim   = cfg.embed_dim
        self.hidden_dim  = cfg.ffn_hidden_dim # per-expert hidden size
        self.use_gate    = cfg.ffn.use_gate
        self.num_layers  = cfg.num_layers
        self.num_experts = cfg.ffn.MoE_num_experts
        self.top_k       = cfg.ffn.MoE_num_experts_per_tkn

        self.capacity_factor = cfg.ffn.MoE_capacity_factor
        self.aux_loss_coef   = cfg.ffn.MoE_router_aux_loss_coef
        self.z_loss_coeff    = cfg.ffn.MoE_router_z_loss_coeff


        if self.num_experts < 2:
            raise ValueError("MoE needs num_experts >= 2 (use GLU/MLP for a single expert)")
        if not 1 <= self.top_k <= self.num_experts:
            raise ValueError("need 1 <= num_experts_per_tkn <= num_experts")


        self.router = nn.Parameter(torch.empty(self.embed_dim, self.num_experts))

        # TODO: 3D Parameter Stacking vs. nn.ModuleList 
        # TODO: Expert Capacity & Vectorized Dispatch (torch.bmm
        # TODO: Load balancing loss

        self._init_weights()

    def _init_weights(self):
        init_custom_weight(self.router, is_residual_output=False)
        for expert in self.experts:
            expert._init_weights()

    def forward(self, x):
        ...

In [ ]:
class Level3_Transformer_Block(nn.Module):

    def __init__(self, cfg:ModelConfig):
        super().__init__()

        self.norm1 = nn.RMSNorm(cfg.embed_dim)
        self.norm2 = nn.RMSNorm(cfg.embed_dim)

        self.attention_sub_block = Attention_SubBlock(cfg)


        if cfg.ffn.use_gate == False:
            self.ffn_sub_block = MLP_SubBlock()

        if cfg.ffn.use_gate and cfg.ffn.num_experts == 0:
            self.ffn_sub_block = GLU_SubBlock()

        if cfg.ffn.num_experts > 0:
            self.ffn_sub_block = Mixture_of_Experts_SubBlock()
        
        ...

    def forward(
        self,
        x: torch.Tensor,
        cos: torch.Tensor,
        sin: torch.Tensor,
        cache: Optional[KV_cache] = None,
        layer_idx: int = 0,

        lane_ids =None, 
        attn_mask=None

    ) -> torch.Tensor:
        
        residual = x
        
        x = self.norm(x)

        x = self.attention_sub_block(
            x, 
            cos, 
            sin, 
            cache=cache, 
            layer_idx=layer_idx,
            lane_ids=lane_ids, 
            attn_mask=attn_mask,
        )

        ffn_residual  = x  

In [ ]:
    class Level3_FeedForward_Model(nn.Module):

        def __init__(self, cfg:ModelConfig):
            super().__init__()

            self.vocab_size = cfg.vocab_size
            self.embed_dim  = cfg.embed_dim  

            self.tied_embeddings = cfg.tied_embeddings      


            self.embed  = nn.Parameter(torch.empty(self.vocab_size, self.embed_dim))
            
            if not self.tied_embeddings:
                self.unembed = nn.Parameter(torch.empty(self.embed_dim, self.vocab_size))


            self.rotary = RotaryEmbedding(cfg)

            self.layers = nn.ModuleList([Level3_Transformer_Block(cfg) 
                                        for _ in range(cfg.num_layers)
                                        ])

            self.final_rms_norm = nn.RMSNorm(cfg.embed_dim)


            self.init_params()



        def init_params(self):
            init_custom_weight(self.embed,   is_residual_output=False)
            
            if not self.tied_embeddings:
                init_custom_weight(self.unembed, is_residual_output=False)


        def forward(
                self, 
                input_ids: torch.Tensor, 
                cache: Optional[KV_cache] = None,
                lane_ids: Optional[torch.Tensor] = None,  # which cache lane each batch row uses
                attn_mask=None, # attn_mask only used without cache (tests)
        ) -> torch.Tensor:

            batch_size, seq_len = input_ids.shape

            device = input_ids.device # from the data, never a global
            # print('input_ids.device -> ', input_ids.device)

            # [batch_size, seq_len] -> [batch_size, seq_len, embed_dim]
            x = self.embed[input_ids] 



            if cache is None:
                position_ids = torch.arange(seq_len, device=device).expand(batch_size, -1)
                # attn_mask stays whatever the caller passed (None -> is_causal fast path)
            else:
                if lane_ids is None:
                    lane_ids = torch.arange(batch_size, device=device)
                position_ids, attn_mask = cache.prepare(lane_ids, seq_len)


            cos, sin = self.rotary(position_ids, dtype=x.dtype)


            
            for i, layer in enumerate(self.layers):
                    x = layer(x, 
                            cos, 
                            sin, 
                            cache=cache, 
                            layer_idx=i,
                            lane_ids=lane_ids,
                            attn_mask=attn_mask,
                            )        

            if cache is not None:
                cache.commit(lane_ids, seq_len) # once, after ALL layers


            x = self.final_rms_norm(x)


            if self.tied_embeddings:
                return x @ self.embed.T                    # [B, T, V]
            
            return x @ self.unembed


In [ ]:
TOTAL_STEPS = 3000 # the WHOLE planned run, across all stages


# Same schedule args in every stage (see the LR section below)
sched = dict(schedule_total_steps=TOTAL_STEPS, warmup_steps=100, min_lr_ratio=0.1)

# Clean slate: remove stale checkpoints from earlier experiments,
# otherwise resume_from="latest" could pick one of them up later
for p in training_engine.list_checkpoints(configs):
    p.unlink()





seed = 42
torch.manual_seed(seed)
model = Level3_FeedForward_Model(configs).to(device)

# Compile the model
# model = torch.compile(
#     model, 
#     mode="max-autotune",
#     dynamic=True, # variable seq_len
#     # fullgraph=True,
# )          


training_engine.train_and_save_model(
    model, configs, device, 1000, seed=seed, **sched
)




model = Level3_FeedForward_Model(configs)

training_engine.train_and_save_model(
    model, configs, device, 2000, seed=seed, resume_from="latest", **sched
)




model = Level3_FeedForward_Model(configs)

training_engine.train_and_save_model(
    model, configs, device, 3000, seed=seed, resume_from="latest", **sched
)


training_engine.plot_loss_history(configs)





# small blip at steps 999, 1999 and 2999. The last evaluation of each stage uses the full validation set, while the others use 20 batches
import matplotlib.pyplot as plt

rows = [l.strip().split(",") for l in open(configs.val_loss_history) if l.strip()]
steps, losses = [int(s) for s, _ in rows], [float(v) for _, v in rows]

plt.style.use("dark_background")
plt.figure(figsize=(10, 5))
plt.plot(steps, losses, color="#00FFCC", label="val loss")
for boundary in (1000, 2000):
    plt.axvline(boundary, color="gray", linestyle=":", label="resume point" if boundary == 1000 else None)
plt.xlabel("Step"); plt.ylabel("Cross-Entropy Loss (Nats)")
plt.grid(True, color="#333333"); plt.legend(); plt.show()




# if torch.cuda.is_available():
#     max_mem_bytes = torch.cuda.max_memory_allocated()
#     max_mem_gb = max_mem_bytes / (1024 ** 3)
#     print(f"Max memory allocated: {max_mem_gb:.2f} GB")


In [ ]:
# Load the JSON file
with open("2048-tokenizer.json", "r", encoding="utf-8") as f:
    tokenizer_data = json.load(f)
eos_token_id = tokenizer_data.get("special_tokens", {}).get("<|endoftext|>")


model = load_model_weights(
    Level3_FeedForward_Model(configs),
    configs.save_path,
    device,
)

prompt_strs = [
    "write a story about",
    "write like a little boy that was angry",
    "write as a little boy that was happy,",
    "my name is ",
] 



results = advanced_inference(
    model=model, 
    configs=configs,
    prompt_strs=prompt_strs,
    device=device, 
    max_new_tokens=500,     # far beyond max_context_window=64: fine
    max_KV_lanes=2, 
    chunk_size=16, 
    eos_token_id=eos_token_id,
)

for r in results:
    print("---" * 20); print(r.text, f"[{r.finish_reason}]")
    